# Baliser le corpus avec spaCy

In [1]:
import spacy
import re
import xml.etree.ElementTree as ET
from pathlib import Path
from lxml import etree
from joblib import Parallel, delayed

In [2]:
nlp_fra = spacy.load("fr_core_news_sm")
nlp_ita = spacy.load("it_core_news_sm")

In [6]:
print("\n".join(sorted(nlp_ita.Defaults.stop_words)))

a
abbastanza
abbia
abbiamo
abbiano
abbiate
accidenti
ad
adesso
affinche
agl
agli
ahime
ahimè
ai
al
alcuna
alcuni
alcuno
all
alla
alle
allo
allora
altri
altrimenti
altro
altrove
altrui
anche
ancora
anni
anno
ansa
anticipo
assai
attesa
attraverso
avanti
avemmo
avendo
avente
aver
avere
averlo
avesse
avessero
avessi
avessimo
aveste
avesti
avete
aveva
avevamo
avevano
avevate
avevi
avevo
avrai
avranno
avrebbe
avrebbero
avrei
avremmo
avremo
avreste
avresti
avrete
avrà
avrò
avuta
avute
avuti
avuto
basta
bene
benissimo
brava
bravo
c'
casa
caso
cento
certa
certe
certi
certo
che
chi
chicchessia
chiunque
ci
ciascuna
ciascuno
cima
cio
cioe
circa
citta
città
co
codesta
codesti
codesto
cogli
coi
col
colei
coll
coloro
colui
come
cominci
comunque
con
concernente
conciliarsi
conclusione
consiglio
contro
cortesia
cos
cosa
cosi
così
cui
d'
da
dagl
dagli
dai
dal
dall
dall'
dalla
dalle
dallo
dappertutto
davanti
degl
degli
dei
del
dell
dell'
della
delle
dello
dentro
detto
deve
di
dice
dietro
dire
dirimpetto


In [15]:
DIR_V0 = Path('output/v0/')
OUTPUT_DIR = Path('output/Vspacy/')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True) # Créer le dossier de sortie s'il n'existe pas déjà

LABEL_MAP = {
    "PER": "persName",
    "LOC": "placeName",
    "GPE": "placeName",
}

In [16]:
def detect_lang(text):
    fr_markers = len(re.findall(r'\b(le|la|les|de|du|des|un|une|et|en|je|il|elle|nous|vous|ils)\b', text, re.IGNORECASE))
    it_markers = len(re.findall(r'\b(il|la|le|di|del|della|un|una|e|in|io|lui|lei|noi|voi|loro)\b', text, re.IGNORECASE))
    return "it" if it_markers > fr_markers else "fr"

def annotate_text(text, nlp_fra, nlp_ita):
    if not text or not text.strip():
        return [{"text": text, "tag": None}]

    nlp = nlp_ita if detect_lang(text) == "it" else nlp_fra
    doc = nlp(text)

    segments = []
    last = 0
    for ent in doc.ents:
        tag = LABEL_MAP.get(ent.label_)
        if not tag:
            continue
        if ent.start_char > last:
            segments.append({"text": text[last:ent.start_char], "tag": None})
        segments.append({"text": ent.text, "tag": tag})
        last = ent.end_char
    segments.append({"text": text[last:], "tag": None})
    return segments

def rebuild_text(element, attr, segments, insert_start=0):
    if not any(s["tag"] for s in segments):
        return 0

    if attr == "text":
        element.text = None
    else:
        # Effacer la queue de l'enfant précédent
        if insert_start > 0:
            try:
                element[insert_start - 1].tail = None
            except (IndexError, TypeError):
                pass

    inserted = 0
    for seg in segments:
        if seg["tag"]:
            new_el = etree.Element(seg["tag"])
            new_el.text = seg["text"]
            new_el.tail = ""
            element.insert(insert_start + inserted, new_el)
            inserted += 1
        else:
            pos = insert_start + inserted
            if pos == 0:
                element.text = (element.text or "") + seg["text"]
            else:
                try:
                    element[pos - 1].tail = (element[pos - 1].tail or "") + seg["text"]
                except (IndexError, TypeError):
                    element.text = (element.text or "") + seg["text"]
    return inserted

def process_element(root, nlp_fra, nlp_ita):
    all_elements = []
    stack = [root]
    while stack:
        el = stack.pop()
        all_elements.append(el)
        for child in reversed(list(el)):
            stack.append(child)

    for element in all_elements:
        try:
            # Snapshot des enfants AVANT toute modification
            original_children = list(element)

            # 1. Traiter element.text
            if element.text and element.text.strip():
                segments = annotate_text(element.text, nlp_fra, nlp_ita)
                rebuild_text(element, "text", segments, insert_start=0)

            # 2. Traiter uniquement les tails des enfants originaux
            for child in original_children:
                if child.tail and child.tail.strip():
                    # Position actuelle après les insertions précédentes
                    idx = list(element).index(child)
                    segments = annotate_text(child.tail, nlp_fra, nlp_ita)
                    child.tail = None
                    rebuild_text(element, "tail", segments, insert_start=idx + 1)

        except Exception as e:
            import traceback
            print(f"  → Erreur sur élément <{element.tag}> : {e}")
            traceback.print_exc()
            continue

def process_file(xml_path, output_dir):
    """Traite un fichier XML dans son propre worker : charge les modèles localement."""
    nlp_fra = spacy.load("fr_core_news_sm")
    nlp_ita = spacy.load("it_core_news_sm")

    try:
        tree = etree.parse(str(xml_path), etree.XMLParser(remove_blank_text=False))
        process_element(tree.getroot(), nlp_fra, nlp_ita)
        output_path = output_dir / xml_path.name
        tree.write(str(output_path), encoding="utf-8", xml_declaration=True, pretty_print=True)
        return f"✓ {xml_path.name}"
    except Exception as e:
        return f"✗ {xml_path.name} — erreur : {e}"



In [17]:
# --- Exécution parallèle ---
xml_files = list(DIR_V0.glob('*.xml'))
print(f"{len(xml_files)} fichiers à traiter avec 4 workers...\n")

results = Parallel(n_jobs=4, backend="loky", verbose=0)(
    delayed(process_file)(f, OUTPUT_DIR) for f in xml_files
)

for r in results:
    print(r)
print("\nTerminé.")

19 fichiers à traiter avec 4 workers...

✓ Agucchi_TrattatoPittura.xml
✓ Daret_VieRaphael.xml
✓ DupuyDuGrez_TraitePeinture.xml
✓ Freart_IdeaDellaPerfezione.xml
✓ Lomazzo_Idea.xml
✓ Lomazzo_TraicteProportion.xml
✓ Marino_DicerieSacre.xml
✓ Monier_HistoireArtsRapportDessein.xml
✓ Pader_LaPeintureParlante.xml
✓ Pader_SongeEnigmatique.xml
✓ Piles_AbregeViePeintres.xml
✓ Piles_ConversationsConnaissancePeinture.xml
✓ Piles_CoursPeinture.xml
✓ Piles_DialogueColoris.xml
✓ Vinci_TraitePeinture_fra.xml
✓ Vinci_TrattatoPittura_ITA.xml
✓ Zuccari_IdeaPittori.xml
✓ Zuccari_Lettera.xml
✓ Zuccari_OrigineProgressoAcademiaDissegno.xml

Terminé.
